# Random Forest

Random forest adalah kumpulan pohon keputusan yang dilatih pada sampel bootstrap dan subset fitur acak. Pengacakan fitur menurunkan korelasi antarpohon, sehingga gabungan prediksinya lebih stabil daripada bagging biasa.

## Tujuan

- Memahami bootstrap sampling dan pemilihan fitur acak.
- Membandingkan bagging dengan random forest.
- Menerapkan random forest pada data churn dan data sel tumor.
- Mencari hyperparameter dengan grid search.


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample


In [ ]:
def get_accuracy(X_train, X_test, y_train, y_test, model):
    return {
        "test Accuracy": metrics.accuracy_score(y_test, model.predict(X_test)),
        "train Accuracy": metrics.accuracy_score(y_train, model.predict(X_train)),
    }


def tree_correlation(X_test, model):
    predictions = pd.DataFrame(
        {
            f"estimator {index + 1}": tree.predict(X_test.to_numpy())
            for index, tree in enumerate(model.estimators_)
        }
    )
    correlation = predictions.corr()
    print("Average correlation between predictors:",
          round(correlation.mean().mean() - 1 / len(model.estimators_), 4))
    return correlation


## Bootstrap sampling

Bootstrap mengambil sampel dengan pengembalian. Satu baris bisa terpilih lebih dari sekali, dan sebagian baris lain tidak terpilih sama sekali.


In [ ]:
CHURN_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "ChurnData.csv"
)
churn_data = pd.read_csv(CHURN_URL)
churn_data = churn_data[
    ["tenure", "age", "address", "income", "ed", "employ", "equip",
     "callcard", "wireless", "churn"]
]
churn_data["churn"] = churn_data["churn"].astype("int")
churn_data.head()


In [ ]:
for _ in range(3):
    print(resample(churn_data[0:5]).index.tolist())


## Pemilihan fitur acak

Setiap simpul hanya mempertimbangkan sebagian fitur. Untuk klasifikasi, akar kuadrat dari jumlah fitur adalah pilihan yang lazim.


In [ ]:
features = churn_data[["tenure", "age", "address", "income", "ed", "employ", "equip"]]
target = churn_data["churn"]

feature_count = features.shape[1]
max_features = round(np.sqrt(feature_count))
print(f"Jumlah fitur: {feature_count}, fitur per simpul: {max_features}")

for _ in range(3):
    print(random.sample(range(feature_count), max_features))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=1
)
print("Train set", X_train.shape, y_train.shape)
print("Test set", X_test.shape, y_test.shape)


## Bagging sebagai pembanding

Bagging melatih banyak pohon pada sampel bootstrap, tetapi memakai seluruh fitur di setiap simpul.


In [ ]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=2),
    n_estimators=20,
    random_state=0,
    bootstrap=True,
).fit(X_train, y_train)
print(get_accuracy(X_train, X_test, y_train, y_test, bagging_model))


In [ ]:
tree_correlation(X_test, bagging_model)


Korelasi antarpohon yang tinggi membuat bagging kurang efektif. Random forest menurunkan korelasi itu lewat pemilihan fitur acak.


In [ ]:
forest_model = RandomForestClassifier(
    max_features=max_features, n_estimators=50, random_state=0, n_jobs=-1
).fit(X_train, y_train)
print(get_accuracy(X_train, X_test, y_train, y_test, forest_model))


In [ ]:
tree_correlation(X_test, forest_model)


## Contoh data sel tumor

Data sampel sel dipakai untuk menguji random forest pada klasifikasi tumor jinak dan ganas.


In [ ]:
CANCER_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "cell_samples.csv"
)
cell_data = pd.read_csv(CANCER_URL)
cell_data = cell_data[pd.to_numeric(cell_data["BareNuc"], errors="coerce").notnull()]

cell_features = cell_data[
    ["Clump", "UnifSize", "UnifShape", "MargAdh", "SingEpiSize",
     "BareNuc", "BlandChrom", "NormNucl", "Mit"]
].astype("int")
cell_target = cell_data["Class"].astype("int")

X_train, X_test, y_train, y_test = train_test_split(
    cell_features, cell_target, test_size=0.2, random_state=4
)
print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)


In [ ]:
param_grid = {
    "n_estimators": [25, 50],
    "max_depth": [5, 10],
    "max_features": ["sqrt", "log2"],
}

search = GridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
search.best_score_


In [ ]:
search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, search.best_estimator_))


## Latihan: klasifikasi jenis obat

Data resep pasien dipakai untuk memprediksi jenis obat. Fitur kategorik disandikan lebih dulu.


In [ ]:
DRUG_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "drug200.csv"
)
drug_data = pd.read_csv(DRUG_URL, delimiter=",")
drug_features = drug_data[["Age", "Sex", "BP", "Cholesterol", "Na_to_K"]].copy()

drug_features["Sex"] = drug_features["Sex"].map({"F": 0, "M": 1})
drug_features["BP"] = drug_features["BP"].map({"LOW": 0, "NORMAL": 1, "HIGH": 2})
drug_features["Cholesterol"] = drug_features["Cholesterol"].map(
    {"NORMAL": 0, "HIGH": 1}
)

X_train, X_test, y_train, y_test = train_test_split(
    drug_features, drug_data["Drug"], test_size=0.2, random_state=4
)
X_train.head()


In [ ]:
drug_search = GridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid={"n_estimators": [25, 50], "max_depth": [5, 10]},
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)

drug_search.best_score_, drug_search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, drug_search.best_estimator_))


## Ringkasan

Random forest mengalahkan bagging pada data churn karena korelasi antarpohonnya lebih rendah. Penambahan pohon sampai batas tertentu menurunkan galat tanpa membuat model overfitting.
